# Inspect an execution trace

Use traces to inspect where time was spent in a computation. The first example needs only the [SDK installation](../getting_started/install.md); select that environment's notebook kernel. Neither Experiment nor an Adapter is required.

The displayed durations are observations from this execution, not evidence of comparative performance or improvements.

Start with `%%ommx_trace` to display activity inside a cell. A **span** is a named processing interval with start and end times. Parent-child intervals show where operations such as evaluation were called. OMMX emits this information through [OpenTelemetry](https://opentelemetry.io/) and displays or saves it through {mod}`ommx.tracing`.

- **`%%ommx_trace`**: a Jupyter cell magic displaying spans as a nested text tree and providing a Chrome Trace Event Format JSON download.
- **{class}`~ommx.tracing.capture_trace` / {func}`~ommx.tracing.traced`**: a context manager and decorator for the same mechanism in Python scripts, tests, and CI.

No additional OTel exporter is needed for local display. Configure the optional [external backend](#own-tracer-provider) only if sending traces elsewhere.

## Quick tour

### Cell magic (`%%ommx_trace`)

Load the extension once per notebook, usually in the first cell.


In [1]:
%load_ext ommx.tracing

Then begin any cell with `%%ommx_trace` to capture it.


In [2]:
%%ommx_trace
from ommx import DecisionVariable, Instance, Sense

x = DecisionVariable.binary(0, name="x")
y = DecisionVariable.binary(1, name="y")
instance = Instance.from_components(
    decision_variables=[x, y],
    objective=x + y,
    constraints={},
    sense=Sense.Maximize,
)
solution = instance.evaluate({0: 1.0, 1: 1.0})

The cell displays:

1. A **nested text tree** of spans, with durations and representative attributes.
2. A **download link** for the full Chrome Trace Event Format JSON. Open it in [Perfetto](https://ui.perfetto.dev/), [speedscope](https://www.speedscope.app/), or `chrome://tracing` to inspect a flamegraph.

```{note}
The text tree and download link are a minimal starting point and may evolve, for example into an embedded interactive flamegraph. Treat layout and markup details as unstable.
```

If the cell raises, the trace HTML still renders, with `[ERROR]` markers on failed spans. The exception is then re-raised, so automation such as `nbconvert --execute`, papermill, or pytest-nbval still sees a failed cell.

### Context manager (`capture_trace`)

Use the same mechanism in an ordinary Python script.


In [3]:
from ommx.tracing import capture_trace, save_chrome_trace
from ommx import DecisionVariable, Instance, Sense

x = DecisionVariable.binary(0, name="x")
y = DecisionVariable.binary(1, name="y")

instance = Instance.from_components(
    decision_variables=[x, y],
    objective=x + y,
    constraints={},
    sense=Sense.Maximize,
)

with capture_trace() as trace:
    solution = instance.evaluate({0: 1.0, 1: 1.0})

trace

└── ommx_trace_block (48.0 µs) {scope=ommx.tracing}

`trace` is a {class}`~ommx.tracing.TraceResult` populated when the block exits.

- {attr}`~ommx.tracing.TraceResult.request`: the exported OTLP {class}`~opentelemetry.proto.collector.trace.v1.trace_service_pb2.ExportTraceServiceRequest`.
- {attr}`~ommx.tracing.TraceResult.spans`: a flattened list of exported OTLP protobuf spans.
- {meth}`~ommx.tracing.TraceResult.otlp_protobuf`: protobuf bytes of the OTLP export request, also used for Experiment trace storage.

In a notebook, evaluating `trace` shows the text tree through its representation. For explicit rendering or saving, use:

- {func}`~ommx.tracing.render_text_tree`: the nested text tree used by the magic.
- {func}`~ommx.tracing.chrome_trace_json`: the trace as a JSON string.
- {func}`~ommx.tracing.save_chrome_trace`: write JSON to disk, creating required parent directories.

Even if the block raises, `trace.request` is populated and failed spans receive `[ERROR]` markers. Inspect or save it from an outer `except` or `finally`. The original exception propagates unchanged.


In [4]:
import tempfile
from pathlib import Path

output_path = Path(tempfile.gettempdir()) / "ommx_trace.json"
save_chrome_trace(trace, output_path)

### Decorator (`@traced`)

{func}`~ommx.tracing.traced` wraps the {class}`~ommx.tracing.capture_trace` context manager.


In [5]:
import tempfile
from pathlib import Path

from ommx.tracing import traced

evaluate_output = Path(tempfile.gettempdir()) / "evaluate_trace.json"

@traced(output=str(evaluate_output))
def evaluate_once(inst):
    return inst.evaluate({0: 1.0, 1: 1.0})

solution = evaluate_once(instance)

All three call forms are supported:

```python
@traced
def f(): ...

@traced()
def f(): ...

@traced(name="build_qubo", output="qubo.json")
def f(): ...
```

Behavior:

- Without `name`, the root span uses `fn.__qualname__` to distinguish decorated functions.
- With `output`, a successful call writes Chrome Trace JSON. It also attempts to save on exceptions, but suppresses saving errors so they do not replace the original exception; exception-path saving is best effort.
- `async def` is supported. Coroutine functions are detected with `inspect.iscoroutinefunction` and awaited inside the trace block, so capture does not end immediately after creating the coroutine.

(experiment-run-trace-storage)=
## Optional: Save Run traces in an Experiment

Only this section needs the HiGHS Adapter and Experiment. See [Add HiGHS](../getting_started/install.md#highs-adapter) and the [Experiment tutorial](../tutorial/experiment_management). They are not prerequisites for the earlier investigation. `run.log_solve()` calls the Adapter's `solve()` and records the input and Solution as one Solve in the Run.

The preceding APIs inspect traces locally or save standalone Chrome Trace JSON. To share traces with trial records, enable trace storage on {class}`~ommx.experiment.Experiment`.


In [6]:
from ommx.experiment import Experiment
from ommx_highs_adapter import OMMXHighsAdapter

with Experiment.with_temp_local_registry(store_trace=True) as experiment:
    with experiment.run() as run:
        run.log_parameter("solver", "highs")
        solution = run.log_solve(OMMXHighsAdapter, instance, verbose=False)
        run.log_parameter("objective", solution.objective)

trace = experiment.runs[0].trace
assert trace is not None
trace

└── Run (24.98 ms) {scope=ommx.experiment} [ommx.run.id=0]
    └── solve (8.72 ms) {scope=ommx.adapter.highs} [adapter='ommx_highs_adapter.adapter.OMMXHighsAdapter']
        ├── convert (4.73 ms) {scope=ommx.adapter.highs}
        ├── call (3.46 ms) {scope=ommx.adapter.highs}
        └── decode (414.0 µs) {scope=ommx.adapter.highs}

`store_trace=True` opens a {class}`~ommx.tracing.capture_trace` block named `Run` around each Run context manager created by the Experiment. It measures the lifetime of `with experiment.run() as run:`, not merely the `experiment.run()` method call. On successful Run exit, the completed {class}`~ommx.tracing.TraceResult` is stored in the Artifact as OTLP protobuf bytes. After loading the Experiment, retrieve the same data through {attr}`~ommx.experiment.SealedRun.trace`.


In [7]:
loaded = Experiment.from_artifact(experiment.artifact)
trace = loaded.runs[0].trace
assert trace is not None
trace

└── Run (24.98 ms) {scope=ommx.experiment} [ommx.run.id=0]
    └── solve (8.72 ms) {scope=ommx.adapter.highs} [adapter='ommx_highs_adapter.adapter.OMMXHighsAdapter']
        ├── convert (4.73 ms) {scope=ommx.adapter.highs}
        ├── call (3.46 ms) {scope=ommx.adapter.highs}
        └── decode (414.0 µs) {scope=ommx.adapter.highs}

Stored traces are deliberately scoped to Runs:

- They are not stored by default. Without `store_trace=True`, spans may still reach an active OpenTelemetry provider, but no trace payload is stored in the Artifact.
- The storage boundary is `with experiment.run() as run:`. It excludes the entire Experiment lifetime, manual `commit()`, and waits between notebook cells.
- `store_trace=True` requires using Run as a context manager. Trace storage finalizes at scope exit, so creating a Run and manually calling `finish()` is rejected in this mode.
- Forking a committed Experiment retains existing Run traces. To store traces for newly added Runs too, also pass `store_trace=True` to {meth}`~ommx.experiment.Experiment.fork`.

The text tree summarizes span names, durations, instrumentation scope, and key attributes. For example, `Run` is the Run context's lifetime, `solve` is an Adapter API call, and `convert` / `call` / `decode` are Adapter stages. `{scope=...}` identifies the package or module producing a span; attributes such as `adapter` and `ommx.run.id` identify backends or Runs.

(own-tracer-provider)=
## Send traces to an external OTel backend

Run trace storage is independent of external OTel export configuration. Only this optional example needs the extra package `opentelemetry-exporter-otlp-proto-grpc`. Install it in your SDK environment with `python -m pip install opentelemetry-exporter-otlp-proto-grpc`. Configure the OTLP endpoint and authentication for your external service. Local inspection above needs neither this package nor an endpoint.

To export to OTLP, Jaeger, Honeycomb, or another backend, configure an SDK TracerProvider and exporter before the first OMMX call:

```python
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter()))
trace.set_tracer_provider(provider)

from ommx import Instance
```

`%%ommx_trace`, {class}`~ommx.tracing.capture_trace`, and Experiment trace storage add their collector to the active SDK provider. The same spans can therefore appear in the notebook and an external backend. Configure the provider at the start of the script or kernel, before importing or calling OMMX.
